# Exploratory & Market Analysis
### Project Objectives & Analytical Questions
1. **Portfolio Concentration:** How concentrated is the pharmaceutical product catalog among top-tier manufacturers versus smaller producers? Do top manufacturers dominate complex dosage forms?
2. **Price Variance:** Controlling for identical generic molecule, dosage form and strength, how much price dispersion exists across competing brands? Do top-tier manufacturers consistently charge a brand premium?
3. **Market Gaps:** Which clinical indications and drug classes are hyper-saturated (high brand-to-generic saturation ratio) versus underserved (low brand competition / unmanufactured generic gaps)?

In [9]:
import pandas as pd
import numpy as np

data_dir = '../data/processed'
df = pd.read_csv(f'{data_dir}/master_medicines.csv')
unmanu_generics = pd.read_csv(f'{data_dir}/unmanufactured_generics.csv')

# print(df['type'].unique())

kpis = {
    'Total Medicine Brands': len(df),
    'Allopathic Brands': (df['type'] == 'allopathic').sum(),
    'Herbal Brands': (df['type'] == 'herbal').sum(),
    'Active Manufacturers': df['manufacturer'].nunique(),
    'Active Generic Molecules': df['generic'].nunique(),
    'Unmanufactured Generics (Market Gaps)': len(unmanu_generics),
    'Unique Dosage Forms': df['dosage_form'].nunique(),
    'Unique Clinical Indications': df['indication'].nunique(),
    'Unique Drug Classes': df['drug_class'].nunique()
}

kpi_summary = pd.Series(kpis, name='Count').to_frame()
print("BANGLADESH PHARMA MARKET BASELINE")
display(kpi_summary)

display(df.head())

BANGLADESH PHARMA MARKET BASELINE


,Count
Total Medicine Brands,21712
Allopathic Brands,21361
Herbal Brands,351
Active Manufacturers,232
Active Generic Molecules,1661
Unmanufactured Generics (Market Gaps),50
Unique Dosage Forms,112
Unique Clinical Indications,660
Unique Drug Classes,358


,brand_id,brand_name,type,slug,dosage_form,generic,strength,manufacturer,package_container,price_bdt,pricing_basis,generic_id,drug_class,indication
0,4077,A-Cold,allopathic,a-coldsyrup4-mg5-ml,Syrup,Bromhexine Hydrochloride,4 mg/5 ml,ACME Laboratories Ltd.,100 ml bottle: ৳ 40.12,40.12,per_container,145,Cough expectorants & mucolytics,Bronchitis
1,4006,A-Cof,allopathic,a-cofsyrup10-mg30-mg125-mg5-ml,Syrup,Dextromethorphan + Pseudoephedrine + Triprolidine,(10 mg+30 mg+1.25 mg)/5 ml,ACME Laboratories Ltd.,100 ml bottle: ৳ 100.00,100.00,per_container,346,Combined cough suppressants,Non-productive cough
2,6174,A-Clox,allopathic,a-cloxinjection500-mgvial,Injection,Cloxacillin Sodium,500 mg/vial,ACME Laboratories Ltd.,"500 mg vial: ৳ 28.43,(5's pack: ৳ 142.15),",28.43,per_container,292,Penicillinase-resistant penicillins,UTI caused by Staphylococcus Aureus
3,6173,A-Clox,allopathic,a-cloxinjection250-mgvial,Injection,Cloxacillin Sodium,250 mg/vial,ACME Laboratories Ltd.,"250 mg vial: ৳ 20.00,(5's pack: ৳ 100.00),",20.00,per_container,292,Penicillinase-resistant penicillins,UTI caused by Staphylococcus Aureus
4,6172,A-Clox,allopathic,a-cloxpowder-for-suspension125-mg5-ml,Powder for Suspension,Cloxacillin Sodium,125 mg/5 ml,ACME Laboratories Ltd.,100 ml bottle: ৳ 45.00,45.00,per_container,292,Penicillinase-resistant penicillins,UTI caused by Staphylococcus Aureus


## Question 1: Portfolio Concentration
Evaluate how much of the national drug catalog is controlled by the Top 10 industry giants versus smaller manufacturers and test whether dosage form complexity acts as a barrier to entry.

### Manufacturer Portfolio Concentration (Top 10 vs. Market)

In [10]:
total_brands = df['brand_id'].nunique()

# display(df[df['brand_name']=='Napa'][['brand_id', 'brand_name', 'dosage_form', 'strength']])

mfg_summary = (
    df.groupby('manufacturer')
    .agg(
        brand_count=('brand_id', 'count'),
        unique_brand_names=('brand_name', 'nunique'),
        unique_generics=('generic', 'nunique'),
        unique_dosage_forms=('dosage_form', 'nunique')
    )
    .sort_values(by='brand_count', ascending=False)
    .reset_index()
)

top_10 = mfg_summary.head(10)
top_10_share = top_10['brand_count'].sum() / total_brands * 100

print(f"Top 10 manufacturers account for {top_10_share:.2f}% of all brand records.\n")
display(top_10)

Top 10 manufacturers account for 34.09% of all brand records.



,manufacturer,brand_count,unique_brand_names,unique_generics,unique_dosage_forms
0,Incepta Pharmaceuticals Ltd.,1182,704,689,76
1,Square Pharmaceuticals Ltd.,902,582,546,67
2,Opsonin Pharma Ltd.,900,489,462,60
3,Beximco Pharmaceuticals Ltd.,699,417,385,53
4,ACME Laboratories Ltd.,692,431,410,52
5,ACI Limited,681,371,350,55
6,Drug International Ltd.,677,425,418,46
7,Eskayef Pharmaceuticals Ltd.,657,391,360,49
8,Aristopharma Ltd.,512,315,311,56
9,Renata Limited,499,295,273,40


### Market Structure by Manufacturer Tier
Group all the manufacturers into 3 tiers based on portfolio rank: 

Top 10 Giants, Mid-Tier Challengers, Small / Niche Producers

In [11]:
print(f"Total Manufactureres: {df['manufacturer'].nunique()}")

conditions = [
    mfg_summary.index < 10,
    mfg_summary.index < 50
]
choices = ['1. Top 10 Giants', '2. Mid-Tier (Rank 11-50)']

mfg_summary['mfg_tier'] = np.select(conditions, choices, default='3. Small/Niche (Rank 51-232)')

tier_overview = (
    mfg_summary.groupby('mfg_tier')
    .agg(
        manufacturer_count=('manufacturer', 'count'),
        total_brands=('brand_count', 'sum'),
        median_brands_per_mfg=('brand_count', 'median'),
        median_generics_per_mfg=('unique_generics', 'median')
    )
    .reset_index()
)

# display(tier_overview)

tier_overview['total_brand_share_pct'] = (tier_overview['total_brands'] / total_brands * 100).round(2)
display(tier_overview)

Total Manufactureres: 232


,mfg_tier,manufacturer_count,total_brands,median_brands_per_mfg,median_generics_per_mfg,total_brand_share_pct
0,1. Top 10 Giants,10,7401,686.5,397.5,34.09
1,2. Mid-Tier (Rank 11-50),40,8707,161.5,103.0,40.10
2,3. Small/Niche (Rank 51-232),182,5604,14.5,10.0,25.81


### Dosage Form Complexity & Manufacturing Barriers by Tier

In [28]:
# Cell 4: Map mfg_tier to df and check dosage form share across tiers
# df = df.merge(
#     mfg_summary[['manufacturer', 'mfg_tier']], 
#     on='manufacturer', 
#     how='left'
# )
df['mfg_tier'] = df['manufacturer'].map(
    mfg_summary.set_index('manufacturer')['mfg_tier']
)

# display(df.head())

# # Calculate percentage split across tiers for each dosage form
dosage_tier_pct = pd.crosstab(
    df['dosage_form'],
    df['mfg_tier'],
    normalize='index'
) * 100

dosage_tier_pct['total_brands'] = df['dosage_form'].value_counts()

display(dosage_tier_pct.head())

# # # Filter to major dosage forms (>= 100 brands) and sort by Top 10 Giants' share
major_dosage_forms = (
    dosage_tier_pct[dosage_tier_pct['total_brands'] >= 100]
    .sort_values(by='1. Top 10 Giants', ascending=False)
    .round(1)
)

display(major_dosage_forms.head(10))
print("\n ----- Lower barrier to entry -----")
display(major_dosage_forms.tail(10))

mfg_tier,1. Top 10 Giants,2. Mid-Tier (Rank 11-50),3. Small/Niche (Rank 51-232),total_brands
dosage_form,,,,
Capsule,31.046570,38.858287,30.095143,1997
Capsule (Controlled Release),25.000000,50.000000,25.000000,4
Capsule (Delayed Release),18.764302,40.732265,40.503432,437
Capsule (Extended Release),33.333333,66.666667,0.000000,9
Capsule (Modified Release),27.777778,50.000000,22.222222,36


mfg_tier,1. Top 10 Giants,2. Mid-Tier (Rank 11-50),3. Small/Niche (Rank 51-232),total_brands
dosage_form,,,,
Inhalation Capsule,71.6,22.0,6.4,109
SC Injection,54.0,24.4,21.6,213
Tablet (Extended Release),53.4,39.7,6.9,247
IM/IV Injection,45.9,43.3,10.8,610
Tablet (Sustained Release),44.9,37.8,17.3,127
Ointment,44.2,38.0,17.8,242
Ophthalmic Suspension,42.2,50.0,7.8,116
Pediatric Drops,42.2,37.3,20.5,244
Cream,41.2,37.6,21.2,495



 ----- Lower barrier to entry -----


mfg_tier,1. Top 10 Giants,2. Mid-Tier (Rank 11-50),3. Small/Niche (Rank 51-232),total_brands
dosage_form,,,,
Tablet (Delayed Release),30.3,44.3,25.4,122
IM Injection,28.7,50.5,20.7,275
Syrup,27.4,37.8,34.7,1002
Chewable Tablet,25.8,41.7,32.5,283
Injection,24.3,48.1,27.5,189
Oral Suspension,24.3,37.1,38.6,568
Powder for Suspension,23.2,41.7,35.1,1004
Capsule (Delayed Release),18.8,40.7,40.5,437
Tablet (Enteric Coated),18.2,46.6,35.2,307


#### Key Findings: Market Concentration & Manufacturing Barriers (Question 1)
* **Observation:** The top 10 manufacturers account for 34.1% of all registered brands and the top 50 companies control 74.2% of the catalog. Meanwhile, 182 small/niche companies share the remaining 25.8%.
* **Insight:** Dosage form, Top 10 giants dominate at Inhalation Capsules (71.6% share), Extended-Release Tablets (53.4%), and IM/IV Injections (45.9%), where small companies hold less than 11% share. Conversely, small companies crowd into lower-barrier liquids (Syrups 34.7%, Oral Suspensions 38.6%) and Delayed/Timed-Release Capsules (40.5%–55.6%).
* **Business Recommendation:** A mid-tier manufacturer looking to escape fierce price wars with 180+ small competitors should avoid launching more standard syrups or basic gastric capsules. Instead invest in Ophthalmic, Injectable or Inhalation lines where competition is restricted to the top 10–50 players.

## Question 2: Price Variance